# $n$-step Bootstrapping

## Introduction

$n$-step TD methods generalize Monte Carlo methods and one-step TD methods. They form a spectrum with MC methods at one end and one-step TD methods at the other, so one can shift smoothly between the two as needed.

One-step TD methods use the same time step for two purposes: how often the action can be changed, and the time interval over which bootstrapping is done. In many applications, actions should be updated very fast, but bootstrapping works best over a length of time in which a significant and recognizable state change has occurred. $n$-step methods separate these intervals by allowing bootstrapping over multiple steps.

The $n$-step bootstrapping idea is considered here on its own, with eligibility-trace mechanisms postponed until later. This makes it possible to examine the issues that arise in the simpler $n$-step setting.

The chapter first considers the prediction problem: using $n$-step methods to predict returns as a function of state for a fixed policy, that is, to estimate $v_\pi$. It then extends the ideas to action values and control methods.

## 7.1 $n$-step TD Prediction

Monte Carlo methods perform an update for each state based on the entire sequence of observed rewards from that state until the end of the episode. One-step TD methods base the update on one reward and use the estimated value of the state one step later as a proxy for the remaining rewards.

An intermediate method performs an update based on an intermediate number of rewards. A two-step update uses the first two rewards and the estimated value of the state two steps later. In general, an $n$-step update uses the first $n$ rewards and the estimated value of the state $n$ steps later.

For a state $S_t$ and the reward sequence $R_{t+1}, R_{t+2}, \ldots, R_T$, the complete return is

$$
G_t = R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + \cdots + \gamma^{T-t-1}R_T.
$$

The one-step return is the target of the one-step TD update:

$$
G_{t:t+1} = R_{t+1} + \gamma V_t(S_{t+1}),
$$

where $V_t : \mathcal{S} \to \mathbb{R}$ is the estimate at time $t$ of $v_\pi$. The reward term is observed directly, and $V_t(S_{t+1})$ takes the place of the remaining terms of the complete return.

The two-step return delays the estimate by one more reward:

$$
G_{t:t+2} = R_{t+1} + \gamma R_{t+2} + \gamma^2 V_{t+1}(S_{t+2}).
$$

The final term corrects for the absence of the later terms in the complete return. The general $n$-step return is

$$
G_{t:t+n} = R_{t+1} + \gamma R_{t+2} + \cdots + \gamma^{n-1}R_{t+n} + \gamma^n V_{t+n-1}(S_{t+n}),
\tag{7.1}
$$

for all $n,t$ such that $n \geq 1$ and $0 \leq t < T-n$. It uses the actual rewards for the first $n$ steps, then uses the estimated value of the state $n$ steps later for the remaining missing terms. If $t+n \geq T$, then all the missing terms are zero and the $n$-step return reduces to the complete return:

$$
G_{t:t+n} = G_t.
$$

For $n>1$, the $n$-step return involves future rewards and states that are not available at time $t$. The value $V_t(S_t)$ therefore cannot be updated using $G_{t:t+n}$ until $R_{t+n}$ has been observed and $V_{t+n-1}$ has been computed. The state-value learning algorithm for the $n$-step return is

$$
V_{t+n}(S_t) = V_{t+n-1}(S_t) + \alpha \left[G_{t:t+n} - V_{t+n-1}(S_t)\right], \qquad 0 \leq t < T,
\tag{7.2}
$$

while all other state values remain unchanged:

$$
V_{t+n}(s) = V_{t+n-1}(s), \qquad \text{for all } s \neq S_t.
$$

In pseudocode, $G_{t:t+n}$ is abbreviated as $G$. The algorithm stores and accesses $S_t$ and $R_t$ using index modulo $n+1$.

```text
n-step TD for estimating V ~= v_pi

Input: a policy pi
Algorithm parameters: step size alpha in (0, 1], a positive integer n
Initialize V(s) arbitrarily, for all s in S
Store and access S_t and R_t using index modulo n + 1

Loop for each episode:
    Initialize and store S_0 != terminal
    T <- infinity
    Loop for t = 0, 1, 2, ...:
        If t < T:
            Take an action according to pi(.|S_t)
            Observe and store the next reward as R_{t+1} and the next state as S_{t+1}
            If S_{t+1} is terminal, then T <- t + 1
        tau <- t - n + 1
        If tau >= 0:
            G <- sum_{i=tau+1}^{min(tau+n,T)} gamma^{i-tau-1} R_i
            If tau + n < T, then G <- G + gamma^n V(S_{tau+n})
            V(S_tau) <- V(S_tau) + alpha [G - V(S_tau)]
    Until tau = T - 1
```

The $n$-step return uses $V_{t+n-1}$ to correct for the missing rewards beyond $R_{t+n}$. Its important property is that its expectation is guaranteed to be a better estimate of $v_\pi$ than $V_{t+n-1}$, in a worst-state sense:

$$
\max_s \left| \mathbb{E}_\pi[G_{t:t+n} \mid S_t=s] - v_\pi(s) \right|
\leq
\gamma^n \max_s \left| V_{t+n-1}(s) - v_\pi(s) \right|,
\tag{7.3}
$$

for all $n \geq 1$. This is the error reduction property of $n$-step returns. It guarantees that all $n$-step TD methods converge to the correct predictions under appropriate technical conditions, forming a family of methods with one-step TD methods and Monte Carlo methods as extreme members.

## 7.2 $n$-step Sarsa

The same $n$-step methods used for prediction can be combined with Sarsa in a straightforward way to produce an on-policy TD control method. The state values are replaced by action values, and the backups use state-action pairs.

The backup diagram for $n$-step Sarsa uses samples all the way, ending with an action-value estimate. Its return is

$$
G_{t:t+n} = R_{t+1} + \gamma R_{t+2} + \cdots + \gamma^{n-1}R_{t+n} + \gamma^n Q_{t+n-1}(S_{t+n}, A_{t+n}), \qquad n \geq 1,\ 0 \leq t < T-n,
\tag{7.4}
$$

with

$$
G_{t:t+n} = G_t \quad \text{if } t+n \geq T.
$$

The natural algorithm updates the estimate of the state-action pair at time $t$ toward this return:

$$
Q_{t+n}(S_t,A_t) = Q_{t+n-1}(S_t,A_t) + \alpha\left[G_{t:t+n} - Q_{t+n-1}(S_t,A_t)\right], \qquad 0 \leq t < T,
\tag{7.5}
$$

while the values of all other state-action pairs remain unchanged:

$$
Q_{t+n}(s,a) = Q_{t+n-1}(s,a), \qquad \text{for all } (s,a) \neq (S_t,A_t).
$$

In the control algorithm, the policy is $\epsilon$-greedy with respect to $Q$, or follows a fixed given policy. The estimate for $S_t,A_t$ is updated after the corresponding $n$-step return is available.

```text
n-step Sarsa for estimating Q ~= q_pi or q_*

Initialize Q(s,a) arbitrarily, for all s in S, a in A
Initialize pi to be epsilon-greedy with respect to Q, or to a fixed given policy
Algorithm parameters: step size alpha in (0,1], small epsilon > 0, positive integer n
Store and access S_t, A_t, and R_t using index modulo n + 1

Loop for each episode:
    Initialize and store S_0 != terminal
    Select and store an action A_0 ~ pi(.|S_0)
    T <- infinity
    Loop for t = 0, 1, 2, ...:
        If t < T:
            Take action A_t
            Observe and store the next reward as R_{t+1} and the next state as S_{t+1}
            If S_{t+1} is terminal, then T <- t + 1
            else select and store an action A_{t+1} ~ pi(.|S_{t+1})
        tau <- t - n + 1
        If tau >= 0:
            G <- sum_{i=tau+1}^{min(tau+n,T)} gamma^{i-tau-1} R_i
            If tau + n < T, then G <- G + gamma^n Q(S_{tau+n}, A_{tau+n})
            Q(S_tau,A_tau) <- Q(S_tau,A_tau) + alpha [G - Q(S_tau,A_tau)]
            If pi is being learned, then ensure that pi(.|S_tau) is epsilon-greedy wrt Q
    Until tau = T - 1
```

For Expected Sarsa, the backup is like $n$-step Sarsa except that its last element is a branch over all possible weighted actions, each weighted by its probability under $\pi$. This is described by the same equation as $n$-step Sarsa, except with the $n$-step return redefined as

$$
G_{t:t+n} = R_{t+1} + \cdots + \gamma^{n-1}R_{t+n} + \gamma^n \bar{V}_{t+n-1}(S_{t+n}), \qquad t+n < T,
\tag{7.7}
$$

with $G_{t:t+n}=G_t$ for $t+n \geq T$. The expected approximate value of a state under the target policy is

$$
\bar{V}_t(s) = \sum_a \pi(a|s)Q_t(s,a), \qquad \text{for all } s \in \mathcal{S}.
\tag{7.8}
$$

If $s$ is terminal, then its expected approximate value is defined to be 0.

## 7.3 $n$-step Off-policy Learning

Off-policy learning learns the value function for one policy, $\pi$, while following another policy, $b$. To use data from $b$, the return is weighted by the relative probability under the two policies of taking the actions that were taken.

For an off-policy version of $n$-step TD, the update for time $t$, made at time $t+n$, is weighted by $\rho_{t:t+n-1}$:

$$
V_{t+n}(S_t) = V_{t+n-1}(S_t) + \alpha \rho_{t:t+n-1}\left[G_{t:t+n} - V_{t+n-1}(S_t)\right], \qquad 0 \leq t < T.
\tag{7.9}
$$

The importance sampling ratio is the relative probability under the two policies of taking the $n$ actions from $A_t$ to $A_{t+n-1}$:

$$
\rho_{t:h} = \prod_{k=t}^{\min(h,T-1)} \frac{\pi(A_k|S_k)}{b(A_k|S_k)}.
\tag{7.10}
$$

If one of the actions would never be taken by $\pi$, then the $n$-step return is given zero weight and is ignored. If an action is much more likely under $\pi$ than under $b$, then this increases the weight that should be given to the return.

For $n$-step Sarsa, the same idea applies, but the action at time $t$ is already known, so it does not need to be corrected. The off-policy update is

$$
Q_{t+n}(S_t,A_t) = Q_{t+n-1}(S_t,A_t) + \alpha \rho_{t+1:t+n-1}\left[G_{t:t+n} - Q_{t+n-1}(S_t,A_t)\right],
\tag{7.11}
$$

for $0 \leq t < T$. The importance sampling ratio starts and ends one step later than in $n$-step TD because the update is for a state-action pair. The selected action is already known and does not have to be sampled, so importance sampling is needed only for later actions.

```text
Off-policy n-step Sarsa for estimating Q ~= q_pi or q_*

Input: an arbitrary behavior policy b such that b(a|s) > 0, for all s in S, a in A
Initialize Q(s,a) arbitrarily, for all s in S, a in A
Initialize pi to be greedy with respect to Q, or to a fixed target policy
Algorithm parameters: step size alpha in (0,1], positive integer n
Store and access S_t, A_t, and R_t using index modulo n + 1

Loop for each episode:
    Initialize and store S_0 != terminal
    Select and store an action A_0 ~ b(.|S_0)
    T <- infinity
    Loop for t = 0, 1, 2, ...:
        If t < T:
            Take action A_t
            Observe and store the next reward as R_{t+1} and the next state as S_{t+1}
            If S_{t+1} is terminal, then T <- t + 1
            else select and store an action A_{t+1} ~ b(.|S_{t+1})
        tau <- t - n + 1
        If tau >= 0:
            rho <- product_{i=tau+1}^{min(tau+n-1,T-1)} pi(A_i|S_i) / b(A_i|S_i)
            G <- sum_{i=tau+1}^{min(tau+n,T)} gamma^{i-tau-1} R_i
            If tau + n < T, then G <- G + gamma^n Q(S_{tau+n}, A_{tau+n})
            Q(S_tau,A_tau) <- Q(S_tau,A_tau) + alpha rho [G - Q(S_tau,A_tau)]
            If pi is being learned, then ensure that pi(.|S_tau) is greedy wrt Q
    Until tau = T - 1
```

The off-policy version of $n$-step Expected Sarsa uses the same update as above except that the importance sampling ratio has one less factor than for Sarsa: it uses $\rho_{t+1:t+n-2}$ instead of $\rho_{t+1:t+n-1}$. The expected approximate value uses $\pi$ over all possible actions in the last state, so the action actually taken there has no effect and does not have to be corrected for.

## 7.4 *Per-decision Methods with Control Variates*

The multi-step off-policy methods from the previous section are simple and conceptually clear, but they are probably not the most efficient. A more sophisticated approach uses per-decision importance sampling ideas.

The ordinary $n$-step return can be written recursively. For the $n$ steps ending at horizon $h$, the return is

$$
G_{t:h} = R_{t+1} + \gamma G_{t+1:h}, \qquad t < h < T,
\tag{7.12}
$$

where

$$
G_{h:h} = V_{h-1}(S_h).
$$

This return is used at time $h$, previously denoted $t+n$.

The first step of the recursion should be weighted by the importance sampling ratio

$$
\rho_t = \frac{\pi(A_t|S_t)}{b(A_t|S_t)}.
$$

If $\pi$ would never select the action actually taken, then $\rho_t=0$. A simple weighting would then make the $n$-step return zero, which can create high variance. Instead, the off-policy return is defined as

$$
G_{t:h} \doteq \rho_t\left(R_{t+1} + \gamma G_{t+1:h}\right) + (1-\rho_t)V_{h-1}(S_t), \qquad t < h < T,
\tag{7.13}
$$

again with $G_{h:h}=V_{h-1}(S_h)$. If $\rho_t=0$, then the target is the same as the estimate and the update causes no change. This means the sample is ignored, leaving the estimate unchanged.

The second term in (7.13) is called a control variate. It does not change the expected update because the importance sampling ratio has expected value one and is uncorrelated with the estimate, so the expected value of the control variate is zero. The off-policy definition in (7.13) generalizes the earlier on-policy definition because in the on-policy case $\rho_t$ is always 1.

With (7.13), the learning rule is the $n$-step TD update in (7.2), which has no explicit importance sampling ratios outside the return.

For action values, the first action does not play a role in the importance sampling. That first action is the one being learned; it does not matter if it was unlikely or impossible under the target policy. Full unit weight must be given to the reward and state that follow it. Importance sampling applies only to the actions that follow.

For action values, the $n$-step on-policy return ending at horizon $h$, in expectation form, can be written recursively as in (7.12), except that the recursion ends with

$$
G_{h:h} = \bar{V}_{h-1}(S_h).
$$

An off-policy form with control variates is

$$
\begin{aligned}
G_{t:h} &\doteq R_{t+1} + \gamma\left(\rho_{t+1}G_{t+1:h} + \bar{V}_{h-1}(S_{t+1}) - \rho_{t+1}Q_{h-1}(S_{t+1},A_{t+1})\right) \\
&= R_{t+1} + \gamma\rho_{t+1}\left(G_{t+1:h} - Q_{h-1}(S_{t+1},A_{t+1})\right) + \gamma\bar{V}_{h-1}(S_{t+1}), \qquad t < h \leq T.
\end{aligned}
\tag{7.14}
$$

If $h<T$, then the recursion ends with

$$
G_{h:h} = Q_{h-1}(S_h,A_h),
$$

whereas, if $h \geq T$, the recursion ends with

$$
G_{T-1:h} = R_T.
$$

The resulting prediction algorithm, after combining with (7.5), is analogous to Expected Sarsa.

Importance sampling often results in high variance updates, forcing the use of a small step-size parameter and causing learning to be slow. Variance might be reduced by taking account of the age of training data, adapting step sizes to the observed variance, or using invariant updates. The next section considers an off-policy learning method that does not use importance sampling.

## 7.5 Off-policy Learning Without Importance Sampling: The $n$-step Tree Backup Algorithm

The tree-backup algorithm is an $n$-step off-policy method that does not use importance sampling. Its backup target combines rewards along the sampled path with estimated values of action nodes that hang off to the side of the sampled path.

In the 3-step tree-backup diagram, the central spine contains sample states, rewards, and sample actions. At each state there are also actions not selected. Because there is no sample data for those unselected actions, the method uses estimates of their values when forming the target for the update.

The update is from the estimated action values of the leaf nodes of the tree. Interior action nodes, corresponding to actions actually taken, do not participate. Each leaf node contributes to the target with a weight proportional to its probability of occurring under the target policy $\pi$.

Thus the first-level action $a$ contributes with weight $\pi(a|S_{t+1})$, except that the action actually taken, $A_{t+1}$, does not contribute at that level. Instead, $\pi(A_{t+1}|S_{t+1})$ weights all the second-level action values. This continues down the tree: action nodes deeper in the tree are weighted by the product of the probabilities of the actions above them under the target policy.

The 3-step tree-backup update can be viewed as six half-steps, alternating between sample half-steps from an action to the subsequent state and expected half-steps that consider all possible actions from that state with their probabilities under the policy.

The one-step return is the same target as Expected Sarsa:

$$
G_{t:t+1} = R_{t+1} + \gamma \sum_a \pi(a|S_{t+1})Q_t(S_{t+1},a),
\tag{7.15}
$$

for $t<T-1$. The two-step tree-backup return is

$$
\begin{aligned}
G_{t:t+2} &= R_{t+1} + \gamma \sum_{a \neq A_{t+1}} \pi(a|S_{t+1})Q_{t+1}(S_{t+1},a) \\
&\quad + \gamma\pi(A_{t+1}|S_{t+1})\left(R_{t+2} + \gamma \sum_a \pi(a|S_{t+2})Q_{t+1}(S_{t+2},a)\right) \\
&= R_{t+1} + \gamma \sum_{a \neq A_{t+1}} \pi(a|S_{t+1})Q_{t+1}(S_{t+1},a) + \gamma\pi(A_{t+1}|S_{t+1})G_{t+1:t+2},
\end{aligned}
$$

for $t<T-2$. This suggests the recursive definition of the tree-backup $n$-step return:

$$
G_{t:t+n} = R_{t+1} + \gamma \sum_{a \neq A_{t+1}} \pi(a|S_{t+1})Q_{t+n-1}(S_{t+1},a) + \gamma\pi(A_{t+1}|S_{t+1})G_{t+1:t+n},
\tag{7.16}
$$

for $t<T-1$ and $n \geq 2$, with the $n=1$ case handled by (7.15), except for $G_{T-1:T} \doteq R_T$.

This target is then used with the usual action-value update rule from $n$-step Sarsa:

$$
Q_{t+n}(S_t,A_t) \doteq Q_{t+n-1}(S_t,A_t) + \alpha\left[G_{t:t+n} - Q_{t+n-1}(S_t,A_t)\right],
$$

for $0 \leq t<T$, while all other state-action pairs remain unchanged.

```text
n-step Tree Backup for estimating Q ~= q_* or q_pi

Initialize Q(s,a) arbitrarily, for all s in S, a in A
Initialize pi to be greedy with respect to Q, or as a fixed given policy
Algorithm parameters: step size alpha in (0,1], a positive integer n
Store and access operations using index modulo n + 1

Loop for each episode:
    Initialize and store S_0 != terminal
    Choose an action A_0 arbitrarily as a function of S_0; store A_0
    T <- infinity
    Loop for t = 0, 1, 2, ...:
        If t < T:
            Take action A_t; observe and store R_{t+1}, S_{t+1}
            If S_{t+1} is terminal:
                T <- t + 1
            else:
                Choose an action A_{t+1} arbitrarily as a function of S_{t+1}; store A_{t+1}
        tau <- t + 1 - n
        If tau >= 0:
            If t + 1 >= T:
                G <- R_T
            else:
                G <- R_{t+1} + gamma sum_a pi(a|S_{t+1}) Q(S_{t+1},a)
            Loop for k = min(t,T-1) down through tau+1:
                G <- R_k + gamma sum_{a != A_k} pi(a|S_k)Q(S_k,a) + gamma pi(A_k|S_k)G
            Q(S_tau,A_tau) <- Q(S_tau,A_tau) + alpha [G - Q(S_tau,A_tau)]
            If pi is being learned, ensure that pi(.|S_tau) is greedy wrt Q
    Until tau = T - 1
```

## 7.6 *A Unifying Algorithm: $n$-step $Q(\sigma)$*

The action-value algorithms considered so far correspond to different backup diagrams. $n$-step Sarsa has all sample transitions. The tree-backup algorithm has all state-to-action transitions fully branched without sampling. $n$-step Expected Sarsa has all sample transitions except for the last state-action one, which is fully branched with an expected value.

The unifying idea is to decide on each step whether to treat the action as a sample, as in Sarsa, or to consider the expectation over all actions, as in the tree-backup update. If one always chooses to sample, the result is Sarsa. If one never samples, the result is tree backup. Expected Sarsa samples for all steps except the last one.

This choice is controlled by $\sigma_t$. The value $\sigma_t=1$ denotes full sampling, and $\sigma_t=0$ denotes a pure expectation without sampling. The algorithm can vary $\sigma$ from step to step.

The tree-backup $n$-step return can be written as

$$
\begin{aligned}
G_{t:h} &= R_{t+1} + \gamma \sum_{a \neq A_{t+1}} \pi(a|S_{t+1})Q_{h-1}(S_{t+1},a) + \gamma\pi(A_{t+1}|S_{t+1})G_{t+1:h} \\
&= R_{t+1} + \gamma\bar{V}_{h-1}(S_{t+1}) - \gamma\pi(A_{t+1}|S_{t+1})Q_{h-1}(S_{t+1},A_{t+1}) + \gamma\pi(A_{t+1}|S_{t+1})G_{t+1:h} \\
&= R_{t+1} + \gamma\pi(A_{t+1}|S_{t+1})\left(G_{t+1:h} - Q_{h-1}(S_{t+1},A_{t+1})\right) + \gamma\bar{V}_{h-1}(S_{t+1}).
\end{aligned}
$$

For $Q(\sigma)$, the action selection probability is multiplied by $1-\sigma_{t+1}$ and the importance-sampling ratio $\rho_{t+1}$ is multiplied by $\sigma_{t+1}$. The $Q(\sigma)$ return is

$$
G_{t:h} = R_{t+1} + \gamma\left(\sigma_{t+1}\rho_{t+1} + (1-\sigma_{t+1})\pi(A_{t+1}|S_{t+1})\right)\left(G_{t+1:h} - Q_{h-1}(S_{t+1},A_{t+1})\right) + \gamma\bar{V}_{h-1}(S_{t+1}),
\tag{7.17}
$$

for $t<h\leq T$. The recursion ends with $G_{h:h}=Q_{h-1}(S_h,A_h)$ if $h<T$, or with $G_{T-1:h}=R_T$ if $h=T$.

The algorithm uses the earlier importance-sampling ratios in the input section because the ratios are incorporated in the $n$-step return.

```text
Off-policy n-step Q(sigma) for estimating Q ~= q_pi or q_*

Input: an arbitrary behavior policy b such that b(a|s) > 0, for all s in S, a in A
Initialize Q(s,a) arbitrarily, for all s in S, a in A
Initialize pi to be greedy with respect to Q, or to a fixed given policy
Algorithm parameters: step size alpha in (0,1], a positive integer n
Store and access operations using index modulo n + 1

Loop for each episode:
    Initialize and store S_0 != terminal
    Choose and store an action A_0 ~ b(.|S_0)
    T <- infinity
    Loop for t = 0, 1, 2, ...:
        If t < T:
            Take action A_t; observe and store R_{t+1}, S_{t+1}
            If S_{t+1} is terminal:
                T <- t + 1
            else:
                Choose and store an action A_{t+1} ~ b(.|S_{t+1})
                Select and store sigma_{t+1}
                Store rho_{t+1} = pi(A_{t+1}|S_{t+1}) / b(A_{t+1}|S_{t+1})
        tau <- t + 1 - n
        If tau >= 0:
            If t + 1 < T:
                G <- Q(S_{t+1},A_{t+1})
            Loop for k = min(t,T-1) down through tau+1:
                If k = T:
                    G <- R_T
                else:
                    Vbar <- sum_a pi(a|S_k)Q(S_k,a)
                    G <- R_k + gamma (sigma_k rho_k + (1 - sigma_k)pi(A_k|S_k))(G - Q(S_k,A_k)) + gamma Vbar
            Q(S_tau,A_tau) <- Q(S_tau,A_tau) + alpha [G - Q(S_tau,A_tau)]
            If pi is being learned, ensure that pi(.|S_tau) is greedy wrt Q
    Until tau = T - 1
```